# 02 – Estimate: test of the preprocessing pipeline

This notebook uses the pipeline in `pipeline.py` (task 19). So far it only checks the expert rule on a small example.
Fitting the pipeline on the real folds, training and choosing the models (task 20) and comparing feature sets
(task 22) follow once the split from task 16 is available.

**What the pipeline does** (fitted on the training part only):

| Branch | Columns | Steps |
|---|---|---|
| numeric | floor_area, floors, dwellings | fill missing with the training median → log → scale |
| categorical | construction_period, building_class, energy_group, heating_system | one 0/1 column per category; rare ones (< 20) merged into "infrequent" |
| municipality | municipality | one 0/1 column per municipality |
| expert rule | construction_period × energy_group | average target of the combination, pulled towards the overall mean (target encoding) → scale |

In [ ]:
import numpy as np
import pandas as pd

from pipeline import split_xy, build_pipeline, ExpertRuleEncoder

data = pd.read_csv("../data/processed/building_clean.csv")
X, y, is_house, groups = split_xy(data)
print("Buildings:", len(X), "| houses:", is_house.sum(), "| groups:", len(np.unique(groups)))
print("Features:", list(X.columns))

## 1. Check the expert rule on a small made-up example

6 made-up buildings, overall mean = 2,400 / 6 = **400**. We use a smoothing of m = 4 so the values are easy to count by hand:
value = (n · mean of combination + 4 · 400) / (n + 4).

In [ ]:
toy_X = pd.DataFrame({
    "construction_period": ["A", "A", "A", "A", "B", "C"],
    "energy_group":        ["gas", "gas", "gas", "gas", "oil", "wood"],
})
toy_y = [200, 200, 200, 200, 1000, 600]

rule = ExpertRuleEncoder(smoothing=4).fit(toy_X, toy_y)
print("Overall mean:", rule.global_mean_)        # expect 400
print(rule.rule_values_)
# A|gas:  n=4, mean 200  -> (4*200 + 4*400) / 8 = 300
# B|oil:  n=1, mean 1000 -> (1000 + 4*400) / 5  = 520   (small cell, pulled strongly towards 400)
# C|wood: n=1, mean 600  -> (600 + 4*400) / 5   = 440

new = pd.DataFrame({"construction_period": ["A", "Z"], "energy_group": ["gas", "gas"]})
print(rule.transform(new).ravel())               # expect [300, 400]: unknown combination gets the overall mean

# without smoothing (m = 0) the rule is just the mean of each combination
print(ExpertRuleEncoder(smoothing=0).fit(toy_X, toy_y).rule_values_)  # expect A|gas 200, B|oil 1000, C|wood 600